In [8]:
import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec,KeyedVectors

In [10]:
import gensim.downloader as api

# 1. Load the pre-trained model (downloads ~1.6GB automatically on first run)
print("Downloading/Loading the Google News Word2Vec model...")
model = api.load('word2vec-google-news-300')

# 2. Extract vector representation
print("\nVector representation of 'man':")
print("Shape:", model['man'].shape)
print(model['man'][:10]) # Printing first 10 values of the 300-dimension array

# 3. Find similar words
print("\nWords similar to 'cricket':")
print(model.most_similar('cricket'))

# 4. Find Similarity Score
print("\nSimilarity between 'man' and 'php':")
print(model.similarity('man', 'php'))

# 5. Odd one out
print("\nOdd one out of [php, java, monkey]:")
print(model.doesnt_match(['php', 'java', 'monkey']))

# 6. Vector Arithmetic (King - Man + Woman)
print("\nKing - Man + Woman:")
print(model.most_similar(positive=['king', 'woman'], negative=['man']))

Downloading/Loading the Google News Word2Vec model...
[==================================================] 100.0% 1662.8/1662.8MB downloaded

Vector representation of 'man':
Shape: (300,)
[ 0.32617188  0.13085938  0.03466797 -0.08300781  0.08984375 -0.04125977
 -0.19824219  0.00689697  0.14355469  0.0019455 ]

Words similar to 'cricket':
[('cricketing', 0.8372225761413574), ('cricketers', 0.8165745735168457), ('Test_cricket', 0.8094819784164429), ('Twenty##_cricket', 0.8068488240242004), ('Twenty##', 0.762426495552063), ('Cricket', 0.7541396617889404), ('cricketer', 0.7372579574584961), ('twenty##', 0.7316356301307678), ('T##_cricket', 0.7304614186286926), ('West_Indies_cricket', 0.698798656463623)]

Similarity between 'man' and 'php':
0.008337103

Odd one out of [php, java, monkey]:
monkey

King - Man + Woman:
[('queen', 0.7118192911148071), ('monarch', 0.6189674735069275), ('princess', 0.5902431011199951), ('crown_prince', 0.5499460697174072), ('prince', 0.5377321243286133), ('kings'

In [11]:
import os
import nltk
import numpy as np
import pandas as pd
import plotly.express as px
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# Download NLTK sentence tokenizer (only needs to be run once)
nltk.download('punkt')

# 1. Read and Preprocess the Data
story = []
data_folder = '/Users/harshraj/Desktop/NLP/data/archive' 

for filename in os.listdir(data_folder):
    if filename.endswith(".txt"):
        filepath = os.path.join(data_folder, filename)
        
        # Read file contents
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            corpus = f.read()
            
        # Tokenize the massive text block into individual sentences
        raw_sentences = nltk.sent_tokenize(corpus)
        
        # Preprocess each sentence into a list of clean, lowercase words
        for sentence in raw_sentences:
            story.append(simple_preprocess(sentence))

print(f"Total sentences extracted: {len(story)}")

# 2. Train the Word2Vec Model
model = Word2Vec(
    window=10,       # Number of context words to look at in both directions
    min_count=2,     # Ignore words appearing less than 2 times in the corpus
    vector_size=100, # Represent each word as an array of 100 numbers
    workers=4        # Number of CPU cores to use for training
)

# Build the vocabulary table
model.build_vocab(story)

# Train the model over 5 epochs
model.train(story, total_examples=model.corpus_count, epochs=5)

# 3. Test the Custom Model
print("\nWords similar to 'daenerys':")
print(model.wv.most_similar('daenerys'))

print("\nOdd one out of the Stark siblings:")
print(model.wv.doesnt_match(['jon', 'robb', 'arya', 'bran', 'rickon', 'sansa']))

# 4. Dimensionality Reduction (PCA) and 3D Visualization
# Limit to the top 200 words to prevent browser lag during plotting
words = model.wv.index_to_key[:200]
vectors = np.array([model.wv[word] for word in words])

# Reduce the 100-dimensional vectors down to 3 dimensions (X, Y, Z)
pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)

# Create a DataFrame and Plot using Plotly
df = pd.DataFrame(vectors_3d, columns=['x', 'y', 'z'])
df['word'] = words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word', title="Game of Thrones Word Embeddings (3D)")
fig.update_traces(textposition='top center')
fig.show()

[nltk_data] Downloading package punkt to /Users/harshraj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Total sentences extracted: 158874


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'



Words similar to 'daenerys':
[('stormborn', 0.8267664909362793), ('targaryen', 0.792833149433136), ('unburnt', 0.7782055139541626), ('queen', 0.7245638966560364), ('princess', 0.6982274651527405), ('rhaegar', 0.6719167232513428), ('elia', 0.6582784056663513), ('viserys', 0.6509697437286377), ('aegon', 0.6486855149269104), ('myrcella', 0.6392821073532104)]

Odd one out of the Stark siblings:
rickon
